## Installs

In [1]:
!pip install datasets
!pip install transformers

## Imports

In [2]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, EarlyStoppingCallback, TrainerCallback
from sklearn.model_selection import train_test_split
from datasets import Dataset, load_dataset

## Data downloading

In [14]:
import pandas as pd

splits = {'train_sft': 'data/train_sft-00000-of-00001.parquet', 'test_sft': 'data/test_sft-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/HuggingFaceTB/everyday-conversations-llama3.1-2k/" + splits["train_sft"])
df

,topic,subtopic,subsubtopic,full_topic,prompt,completion,token_length,messages
0,Shopping,Budgeting,Tracking expenses,Shopping/Budgeting/Tracking expenses,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,163,"[{'content': 'Hey!', 'role': 'user'}, {'conten..."
1,Music,Musical instruments,Instrument history,Music/Musical instruments/Instrument history,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,203,"[{'content': 'Hi', 'role': 'user'}, {'content'..."
2,environmental science,biodiversity,None,environmental science/biodiversity,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,174,"[{'content': 'Hey!', 'role': 'user'}, {'conten..."
3,Shopping,Clothes shopping,Fashion trends,Shopping/Clothes shopping/Fashion trends,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,129,"[{'content': 'Hi', 'role': 'user'}, {'content'..."
4,Cooking,Cooking for others,Catering basics,Cooking/Cooking for others/Catering basics,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,159,"[{'content': 'Hi', 'role': 'user'}, {'content'..."
...,...,...,...,...,...,...,...,...
2255,Home,Rooms in a house,Garage,Home/Rooms in a house/Garage,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,139,"[{'content': 'Hi', 'role': 'user'}, {'content'..."
2256,Fashion,Formal wear,Evening gowns,Fashion/Formal wear/Evening gowns,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,183,"[{'content': 'Hi there', 'role': 'user'}, {'co..."
2257,Food,Grocery shopping strategies,Avoiding impulse buys,Food/Grocery shopping strategies/Avoiding impu...,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,183,"[{'content': 'Hi', 'role': 'user'}, {'content'..."
2258,Health,Mental health,Social connections,Health/Mental health/Social connections,Generate a very simple multi-turn conversation...,User: Hi\n\nAI: Hello! How can I help you toda...,139,"[{'content': 'Hi there', 'role': 'user'}, {'co..."


In [16]:
dataset = Dataset.from_dict({
    "prompt": df.prompt,
    'completion': df.completion})
dataset

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 2260
})

## Data preporating

In [17]:
#MODEL NAME
model_name = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [25]:
def format_example(batch):
    prompts = batch["prompt"]
    completions = batch["completion"]
    texts = [
        f"<System prompt>:\n{p}\n<Generation>:\n{c}"
        for p, c in zip(prompts, completions)
    ]
    return {"text": texts}

In [26]:
dataset = dataset.map(format_example, batched = True)

Map:   0%|          | 0/2260 [00:00<?, ? examples/s]

In [27]:
print(dataset['text'][0])

<System prompt>:
Generate a very simple multi-turn conversation between a User and an AI Assistant about Shopping/Budgeting/Tracking expenses. The conversation should start with a basic greeting like "Hello" or "Hi" and be straightforward. Include 3-4 short exchanges. The AI should give brief, clear answers. The User should ask simple questions.

Start the conversation like this:

User: [Greeting]

AI: Hello! How can I help you today?

User: [Continue with a simple question or statement]

AI: [Respond briefly and clearly]

User: [Ask a follow-up question or make another simple statement]

AI: [Provide a final helpful response]

Make sure the entire conversation remains very simple and easy to understand, focusing on basic topics or requests.
<Generation>:
User: Hi

AI: Hello! How can I help you today?

User: I'm trying to track my expenses. Can you help me with that?

AI: Yes, I can help you track your expenses. You can start by telling me your income and fixed expenses, such as rent a

In [28]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=256,
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/2260 [00:00<?, ? examples/s]

In [29]:
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.2)

train_dataset = tokenized_dataset["train"]
eval_dataset = tokenized_dataset["test"]

train_dataset

Dataset({
    features: ['prompt', 'completion', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 1808
})

## Training process

In [30]:
model = AutoModelForCausalLM.from_pretrained(model_name)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [31]:
from huggingface_hub import login

login(token='YOUR_TOKEN')

In [32]:
repository_name = 'gpt-car-recommender'

class PushToHubEvery5EpochsCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        if state.epoch and int(state.epoch) % 5 == 0:
            kwargs["model"].push_to_hub(repository_name, commit_message=f"Checkpoint at epoch {int(state.epoch)} with cars reviews dataset")
        return control

In [33]:
early_stopping = EarlyStoppingCallback(early_stopping_patience=2)

In [34]:
training_args = TrainingArguments(
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    num_train_epochs=20,
    lr_scheduler_type="cosine",
    report_to="none",
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
)

In [35]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    callbacks=[early_stopping, PushToHubEvery5EpochsCallback()],
)

<ipython-input-35-05222ba6fe91>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [36]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 